# NB7 — MES Top-N Gene Cutoff Sensitivity

Sensitivity analysis for the fixed top-50 gene cutoff used to define each MES module.
Re-scores all eight MES modules using top-25, top-50, and top-100 NMF-weight-ranked genes
and recomputes donor-level (or cell-level) MES–tolerance-positioning Spearman correlations
across all four microglia cohorts.

Addresses **Reviewer 3, Comment 14**: *"Defining signatures using a fixed top-50 gene cutoff
may not accurately reflect factor-specific contribution distributions."*

**Paper:** Zafar SA, Qin W, Liu C, Khan AA, Faisal MS.
*Thymus-derived myeloid programs track microglial tolerance states across human cohorts.* (2026).

**Depends on:** NB3 outputs (`Main_Table1.xlsx` for NMF weights) and NB4/NB5 scored `.h5ad` files.

**Outputs:**
- `Supplementary_TableS24_TopN_Sensitivity.xlsx` (sheets: `long_form`, `pivot_r_by_topN`)
- `Supplementary_FigS24_TopN_Sensitivity.png`

> **Note:** Set `MES_BASE_DIR` to your project root before running.
> Raw data can be obtained from the public repositories listed in Supplementary Table S1.

In [ ]:
from __future__ import annotations

import os, re, time, gc, warnings, traceback
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.stats as scipy_stats
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         8,
    "axes.titlesize":    9,
    "axes.labelsize":    8,
    "xtick.labelsize":   7,
    "ytick.labelsize":   7,
    "legend.fontsize":   7,
    "figure.dpi":        150,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.05,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size":  3,
    "ytick.major.size":  3,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    import seaborn as sns
    sns.set_style("ticks")
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    print("[WARN] seaborn not installed; figures will use matplotlib only.")

import anndata as ad
import scanpy as sc

try:
    from statsmodels.stats.multitest import multipletests
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print("[WARN] statsmodels not installed; BH correction will use scipy fallback.")


# =============================================================================
# 0) PATHS
# =============================================================================

np.random.seed(42)

BASE_DIR  = Path(os.environ.get("MES_BASE_DIR", "."))  # <-- SET PATH
PROC_DIR  = BASE_DIR / "Process Data" / "aim2_microglia"
MANUS_DIR = BASE_DIR / "Manuscript data"
TAB_DIR   = MANUS_DIR / "Tables"
FIG_DIR   = MANUS_DIR / "Figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# NMF weights table produced by NB3
WEIGHTS_PATH_CANDIDATES = [
    BASE_DIR / "Manuscript data" / "Tables" / "Main_Table1.xlsx",
    BASE_DIR / "Manuscript Data" / "Tables" / "Main_Table1.xlsx",
]
WEIGHTS_PATH = next((p for p in WEIGHTS_PATH_CANDIDATES if p.exists()), None)
assert WEIGHTS_PATH is not None, (
    "Main_Table1.xlsx not found. Re-run NB3 first to generate NMF weights."
)

# Scored microglia h5ads produced by NB4/NB5
SCORED_GLOB = str(PROC_DIR / "*__microglia_scored.h5ad")

DPI     = 1200
N_BOOT  = 500    # bootstrap iterations (faster than full pipeline; sufficient for sensitivity)
ALPHA   = 0.05

TOP_N_VARIANTS = [25, 50, 100]
MES_COLS = [f"MES0{i}" for i in range(1, 9)]


# =============================================================================
# 1) LOGGING AND I/O HELPERS
# =============================================================================

def _now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str) -> None:
    print(f"[{_now()}] {msg}", flush=True)

def save_fig(path: Path, fig=None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    (fig or plt).savefig(path, dpi=DPI, bbox_inches="tight")
    if fig is not None:
        plt.close(fig)
    else:
        plt.close()
    log(f"SAVED FIG  {path}")

def save_xlsx_multi(dfs: Dict[str, pd.DataFrame], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in dfs.items():
            if df is not None and not (isinstance(df, pd.DataFrame) and df.empty):
                df.to_excel(w, index=False, sheet_name=name[:31])
    log(f"SAVED TABLE {path}")


# =============================================================================
# 2) STATISTICAL HELPERS
# =============================================================================

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg FDR correction."""
    pvals = np.asarray(pvals, dtype=float)
    if HAS_SM:
        _, q, _, _ = multipletests(pvals, method="fdr_bh")
        return q
    # Fallback: manual BH
    n = len(pvals)
    order = np.argsort(pvals)
    q = np.empty(n)
    q[order] = pvals[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    return np.minimum(q, 1.0)


def safe_corr_with_ci(
    x: pd.Series,
    y: pd.Series,
    method: str = "spearman",
    n_boot: int = N_BOOT,
    seed: int = 42,
) -> Dict:
    """Spearman (or Pearson) correlation with bootstrap 95% CI and permutation p-value."""
    x = pd.to_numeric(x, errors="coerce").to_numpy()
    y = pd.to_numeric(y, errors="coerce").to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 5:
        return {"r": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "p": np.nan, "n": n}
    if method == "spearman":
        r_obs, p_obs = scipy_stats.spearmanr(x, y)
        def _corr(a, b): return scipy_stats.spearmanr(a, b)[0]
    else:
        r_obs, p_obs = scipy_stats.pearsonr(x, y)
        def _corr(a, b): return scipy_stats.pearsonr(a, b)[0]
    rng = np.random.default_rng(seed)
    boots = np.array([
        _corr(x[idx := rng.integers(0, n, n)], y[idx])
        for _ in range(n_boot)
    ])
    boots = boots[np.isfinite(boots)]
    ci_lo = float(np.percentile(boots, 2.5))  if len(boots) else np.nan
    ci_hi = float(np.percentile(boots, 97.5)) if len(boots) else np.nan
    return {"r": float(r_obs), "ci_lo": ci_lo, "ci_hi": ci_hi, "p": float(p_obs), "n": n}


# =============================================================================
# 3) SCORING HELPERS
# =============================================================================

def score_geneset(
    adata: ad.AnnData,
    genes: List[str],
    key: str,
    min_genes: int = 3,
) -> int:
    """Score a gene set using Scanpy score_genes; returns number of genes present."""
    var_upper = adata.var_names.astype(str).str.upper()
    present = [g for g in genes if g.upper() in var_upper]
    if len(present) < min_genes:
        return 0
    # Map back to original case
    upper_to_orig = dict(zip(var_upper, adata.var_names))
    genes_orig = [upper_to_orig[g.upper()] for g in present]
    try:
        sc.tl.score_genes(adata, gene_list=genes_orig, score_name=key, use_raw=False)
    except Exception as e:
        log(f"  score_genes failed for {key}: {e}")
        return 0
    return len(present)


def detect_donor_col(obs: pd.DataFrame) -> Optional[str]:
    """Return the donor/subject ID column name, or None for cell-level only datasets."""
    candidates = [
        "donor_id", "donor", "subject", "SubjectID", "subject_id",
        "individualID", "individual_id", "case_id", "sample",
        "Participant ID", "participant_id",
    ]
    for c in candidates:
        if c in obs.columns:
            n_unique = obs[c].nunique()
            if 2 <= n_unique <= 500:
                return c
    return None


def donor_aggregate(
    obs: pd.DataFrame,
    donor_col: str,
    score_cols: List[str],
) -> pd.DataFrame:
    """Mean-aggregate score columns to donor level."""
    cols = [c for c in score_cols if c in obs.columns]
    return obs.groupby(donor_col)[cols].mean().reset_index()


TOL_HOMEO = ["P2RY12", "CX3CR1", "TMEM119", "GPR34", "SALL1", "CSF1R", "OLFML3"]
TOL_ACTIV = ["APOE", "SPP1", "LPL", "TREM2", "CST7", "CTSD", "TYROBP",
             "FCER1G", "LGALS3", "CD68"]


def prep_tolerance(adata: ad.AnnData) -> None:
    """Ensure tolerance_positioning score is present."""
    if "tolerance_positioning" not in adata.obs.columns:
        score_geneset(adata, TOL_HOMEO, "_tol_h7")
        score_geneset(adata, TOL_ACTIV, "_tol_a7")
        if "_tol_h7" in adata.obs and "_tol_a7" in adata.obs:
            adata.obs["tolerance_positioning"] = (
                adata.obs["_tol_h7"] - adata.obs["_tol_a7"]
            )


# =============================================================================
# 4) LOAD NMF WEIGHTS (from NB3 Main_Table1.xlsx)
# =============================================================================

log("=" * 72)
log("NB7: MES Top-N Gene Cutoff Sensitivity")
log("=" * 72)
log(f"Loading NMF weights from: {WEIGHTS_PATH}")

# Main_Table1.xlsx stores weights with gene as index and MES01..MES08 as columns
df_weights_raw = pd.read_excel(WEIGHTS_PATH, sheet_name=0)

# Identify gene column (first string column)
gene_col = df_weights_raw.columns[0]
df_weights = df_weights_raw.rename(columns={gene_col: "gene"}).copy()
df_weights["gene"] = df_weights["gene"].astype(str).str.upper().str.strip()

mes_weight_cols = [c for c in df_weights.columns if str(c).startswith("MES")]
assert len(mes_weight_cols) == 8, f"Expected 8 MES weight columns, found: {mes_weight_cols}"
log(f"  Weight columns found: {mes_weight_cols}")
log(f"  Genes in weight table: {len(df_weights)}")


# =============================================================================
# 5) LOAD SCORED MICROGLIA COHORTS
# =============================================================================

import glob as _glob

log("Loading scored microglia cohorts ...")
h5ad_files = sorted(_glob.glob(SCORED_GLOB))
assert len(h5ad_files) >= 4, (
    f"Expected >=4 scored .h5ad files in {PROC_DIR}, found {len(h5ad_files)}. "
    "Re-run NB4 and NB5 first."
)

cohorts: Dict[str, ad.AnnData] = {}
for fp in h5ad_files:
    name = Path(fp).stem.replace("__microglia_scored", "")
    log(f"  Reading {name} ...")
    cohorts[name] = sc.read_h5ad(fp)
    gc.collect()

log(f"Loaded {len(cohorts)} cohorts: {list(cohorts.keys())}")


# =============================================================================
# 6) A3 — TOP-N GENE CUTOFF SENSITIVITY
# =============================================================================

log("=" * 72)
log("A3: MES top-N cutoff sensitivity (N = 25, 50, 100)")
log("=" * 72)

try:
    a3_rows = []

    for ds, adata in cohorts.items():
        log(f"  Processing {ds} ...")
        prep_tolerance(adata)

        if "tolerance_positioning" not in adata.obs.columns:
            log(f"    SKIP {ds}: tolerance_positioning not available")
            continue

        donor_col = detect_donor_col(adata.obs)
        all_score_cols = ["tolerance_positioning"]

        # Score each (MES, topN) combination
        for topN in TOP_N_VARIANTS:
            for mes in mes_weight_cols:
                sub = (
                    df_weights[["gene", mes]]
                    .dropna()
                    .sort_values(mes, ascending=False)
                    .head(topN)
                )
                genes = sub["gene"].astype(str).str.upper().tolist()
                col_name = f"{mes}_top{topN}"
                n_present = score_geneset(adata, genes, col_name, min_genes=3)
                if n_present > 0:
                    all_score_cols.append(col_name)

        # Aggregate to donor level if possible
        score_cols_in_obs = [c for c in all_score_cols if c in adata.obs.columns]
        if donor_col is not None:
            ddf = donor_aggregate(adata.obs, donor_col, score_cols_in_obs)
            level = "donor"
        else:
            ddf = adata.obs[score_cols_in_obs].copy()
            level = "cell"

        log(f"    Level: {level} | N units: {len(ddf)}")

        # Correlate each (MES, topN) with tolerance_positioning
        for topN in TOP_N_VARIANTS:
            for mes in mes_weight_cols:
                col_name = f"{mes}_top{topN}"
                if col_name not in ddf.columns:
                    continue
                r = safe_corr_with_ci(
                    ddf["tolerance_positioning"],
                    ddf[col_name],
                    method="spearman",
                    n_boot=N_BOOT,
                )
                a3_rows.append({
                    "dataset":  ds,
                    "level":    level,
                    "MES":      mes,
                    "top_N":    topN,
                    "N":        r["n"],
                    "r":        r["r"],
                    "ci_lo":    r["ci_lo"],
                    "ci_hi":    r["ci_hi"],
                    "p":        r["p"],
                })

        gc.collect()

    df_a3 = pd.DataFrame(a3_rows)
    assert not df_a3.empty, "A3 produced no rows — check cohort loading and scoring."

    df_a3["q_BH"] = bh_fdr(df_a3["p"].fillna(1.0).values)

    # Pivot: per (dataset, MES), show rho at each top_N side by side
    pivot = df_a3.pivot_table(
        index=["dataset", "MES"],
        columns="top_N",
        values="r",
        aggfunc="mean",
    ).reset_index()
    for col in pivot.columns:
        if col not in ("dataset", "MES"):
            pivot.rename(columns={col: f"rho_top{int(col)}"}, inplace=True)
    if "rho_top25" in pivot.columns and "rho_top50" in pivot.columns:
        pivot["delta_25_vs_50"]  = pivot["rho_top25"]  - pivot["rho_top50"]
    if "rho_top100" in pivot.columns and "rho_top50" in pivot.columns:
        pivot["delta_100_vs_50"] = pivot["rho_top100"] - pivot["rho_top50"]

    # Summary statistics
    if "delta_25_vs_50" in pivot.columns:
        med_d25 = pivot["delta_25_vs_50"].abs().median()
        log(f"  Median |Δρ| top-25 vs top-50: {med_d25:.4f}")
    if "delta_100_vs_50" in pivot.columns:
        med_d100 = pivot["delta_100_vs_50"].abs().median()
        log(f"  Median |Δρ| top-100 vs top-50: {med_d100:.4f}")

    # Save tables
    out_table = TAB_DIR / "Supplementary_TableS24_TopN_Sensitivity.xlsx"
    save_xlsx_multi({"long_form": df_a3, "pivot_r_by_topN": pivot}, out_table)

except Exception as e:
    log(f"A3 FAILED: {e}\n{traceback.format_exc()}")
    df_a3, pivot = pd.DataFrame(), pd.DataFrame()


# =============================================================================
# 7) FIGURE — Top-N sensitivity heatmap (one panel per dataset)
# =============================================================================

log("Generating sensitivity figure ...")

try:
    assert not df_a3.empty

    datasets = df_a3["dataset"].unique().tolist()
    n_ds = len(datasets)
    n_col = min(n_ds, 4)
    n_row = int(np.ceil(n_ds / n_col))

    fig, axes = plt.subplots(
        n_row, n_col,
        figsize=(n_col * 3.5, n_row * 3.2),
        squeeze=False,
    )

    CMAP = "RdBu_r"
    VABS = 0.8

    for idx, ds in enumerate(datasets):
        ax = axes[idx // n_col][idx % n_col]
        sub = df_a3[df_a3["dataset"] == ds].copy()
        mat = sub.pivot(index="MES", columns="top_N", values="r")
        mat = mat.reindex(index=mes_weight_cols)
        mat.columns = [f"Top-{c}" for c in mat.columns]

        im = ax.imshow(
            mat.values.astype(float),
            cmap=CMAP, vmin=-VABS, vmax=VABS, aspect="auto",
        )
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=0, fontsize=7)
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index, fontsize=7)
        ax.set_title(ds.replace("_", " "), fontsize=8, pad=4)

        # Annotate each cell with rho value
        for r_i in range(mat.shape[0]):
            for c_i in range(mat.shape[1]):
                val = mat.values[r_i, c_i]
                if np.isfinite(val):
                    ax.text(
                        c_i, r_i, f"{val:.2f}",
                        ha="center", va="center", fontsize=6,
                        color="white" if abs(val) > 0.4 else "black",
                    )

        plt.colorbar(im, ax=ax, shrink=0.7, label="Spearman ρ", pad=0.02)

    # Hide unused axes
    for idx in range(n_ds, n_row * n_col):
        axes[idx // n_col][idx % n_col].set_visible(False)

    fig.suptitle(
        "MES–Tolerance Correlation: Top-N Gene Cutoff Sensitivity (N = 25, 50, 100)",
        fontsize=9, y=1.01,
    )
    plt.tight_layout()
    save_fig(FIG_DIR / "Supplementary_FigS24_TopN_Sensitivity.png", fig)

except Exception as e:
    log(f"FIGURE FAILED: {e}\n{traceback.format_exc()}")


# =============================================================================
# 8) COMPLETION REPORT
# =============================================================================

log("=" * 72)
log("NB7 COMPLETED")
log("=" * 72)
log(f"Tables  : {TAB_DIR}")
log(f"Figures : {FIG_DIR}")
log("")
log("Key outputs:")
log("  Supplementary_TableS24_TopN_Sensitivity.xlsx")
log("    Sheet 'long_form'    — per (dataset, MES, top_N): rho, CI, p, q_BH")
log("    Sheet 'pivot_r_by_topN' — per (dataset, MES): rho at top-25/50/100 + deltas")
log("  Supplementary_FigS24_TopN_Sensitivity.png — heatmap per cohort")
log("")
if not df_a3.empty and "delta_25_vs_50" in pivot.columns:
    med = pivot["delta_25_vs_50"].abs().median()
    log(f"  Median |Δρ| top-25 vs top-50 across all module-cohort pairs: {med:.4f}")
    log(f"  Manuscript claim: 0.063 — {'CONSISTENT' if abs(med - 0.063) < 0.01 else 'CHECK'}")
log("")
log("STATISTICAL METHODS:")
log("  Spearman rank correlation at donor level (cell level for cohorts without donor metadata)")
log("  95% CI: 500-iteration bootstrap")
log(f"  Multiple testing: BH-FDR across all {len(df_a3)} tests")
log("  Top-N gene sets derived from NMF weight-ranked gene list (Main_Table1.xlsx)")
